# AutoPrice — 01. Dataset audit va dastlabki EDA

**Maqsad:** ochiq CarDekho datasetini yuklash, manbasini qayd etish, ustunlarni tekshirish, ma’lumot sifatini baholash va chiroyli grafiklar yaratish.

> Muhim: bu Hindiston bozori datasetidir. Narxlar INR (Indian Rupee). Uni O‘zbekiston bozoridagi UZS narxlarini prognozlash uchun bevosita ishlatmang. Bu dataset faqat Data Science pipeline va metodologiyani o‘rganish uchun.

Manba: [Kaggle — Vehicle Dataset from CarDekho](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho). Kaggle sahifasidagi litsenziya va foydalanish shartlarini yuklab olishdan oldin qayta tekshiring. Manba va litsenziya ma’lumotlarini loyiha hujjatlarida saqlang.


## 0. Muhitni tayyorlash

Google Colab yoki Jupyter’da ishlaydi. Plotly interaktiv 3D grafiklar uchun ishlatiladi.


In [ ]:
%pip -q install pandas numpy matplotlib plotly


In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
plt.rcParams.update({'figure.figsize': (11, 6), 'figure.dpi': 120, 'axes.titlesize': 15, 'axes.labelsize': 11})

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
for folder in (RAW_DIR, PROCESSED_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw data folder:', RAW_DIR.resolve())


## 1. CSV faylni topish va yuklash

Kaggle’dan ZIP/CSV faylni yuklab, `data/raw/` papkasiga joylang. Agar Colab’da ishlasangiz, faylni chap tomondagi Files paneli orqali `/content/data/raw/` ichiga yuklashingiz mumkin. Kod papkadagi CSV fayllarni topadi va qatorlar soni eng kattasini boshlang‘ich nomzod sifatida tanlaydi. Fayl tanlovini albatta tekshiring.


In [ ]:
csv_files = sorted(RAW_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(
        f'CSV topilmadi: {RAW_DIR}. Kaggle’dan CSV faylni yuklab, shu papkaga joylang.'
    )

file_stats = []
for file in csv_files:
    try:
        preview = pd.read_csv(file, nrows=5)
        file_stats.append({'file': file.name, 'columns': len(preview.columns), 'preview_rows': len(preview)})
    except Exception as exc:
        file_stats.append({'file': file.name, 'columns': None, 'preview_rows': f'ERROR: {exc}'})

display(pd.DataFrame(file_stats))
DATA_PATH = csv_files[0]
print('Dastlabki tanlangan fayl:', DATA_PATH.name)

# Agar boshqa CSV kerak bo‘lsa, quyidagi qatorni qo‘lda o‘zgartiring:
# DATA_PATH = RAW_DIR / 'Car details v3.csv'

df_raw = pd.read_csv(DATA_PATH)
print('Dataset hajmi:', df_raw.shape)
display(df_raw.head())


## 2. Dataset haqida birinchi tasavvur

Bu bosqichda ma’lumotni o‘zgartirmaymiz. Avval ustunlar, turlar, bo‘sh qiymatlar va takrorlarni ko‘ramiz.


In [ ]:
print('Qatorlar:', df_raw.shape[0])
print('Ustunlar:', df_raw.shape[1])
print('\nUstun nomlari:')
print(df_raw.columns.tolist())
print('\nData types va non-null counts:')
df_raw.info()


In [ ]:
audit = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'non_null': df_raw.notna().sum(),
    'missing': df_raw.isna().sum(),
    'missing_pct': (df_raw.isna().mean() * 100).round(2),
    'unique_values': df_raw.nunique(dropna=True)
}).sort_values('missing_pct', ascending=False)
display(audit)
print('Takroriy qatorlar:', int(df_raw.duplicated().sum()))


## 3. Ustunlarni standart formatga keltirish

CarDekho datasetining turli versiyalarida ustun nomlari farq qilishi mumkin. Quyidagi kod asosiy ustunlarni yagona nomga keltiradi. Narx valyutasi **INR** sifatida saqlanadi.


In [ ]:
def normalize_column_name(value):
    value = str(value).strip().lower()
    value = re.sub(r'[^a-z0-9]+', '_', value)
    return value.strip('_')

df = df_raw.copy()
df.columns = [normalize_column_name(c) for c in df.columns]

aliases = {
    'car_name': 'name', 'car': 'name', 'vehicle_name': 'name',
    'sellingprice': 'selling_price', 'price': 'selling_price',
    'kms_driven': 'km_driven', 'kilometers_driven': 'km_driven', 'kilometres_driven': 'km_driven',
    'fuel_type': 'fuel', 'seller': 'seller_type', 'gearbox': 'transmission',
    'owner_type': 'owner', 'previous_owners': 'owner'
}
df = df.rename(columns={c: aliases[c] for c in df.columns if c in aliases})

required = {'name', 'year', 'selling_price', 'km_driven'}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f'Kerakli ustunlar topilmadi: {sorted(missing_required)}. Ustun nomlarini tekshiring.')

for col in ['year', 'selling_price', 'km_driven', 'seats']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['name', 'fuel', 'seller_type', 'transmission', 'owner']:
    if col in df.columns:
        df[col] = df[col].astype('string').str.strip()

df['make'] = df['name'].astype('string').str.split().str[0].str.title()
df['model'] = df['name'].astype('string').str.split(n=1).str[1].fillna('Unknown').str.strip()
df['currency'] = 'INR'
df['price_type'] = 'asking_price_or_listed_selling_price'

display(df.head())
print('Standartlashtirilgan ustunlar:', df.columns.tolist())


## 4. Sifat nazorati va tozalashdan oldingi hisobot

Noto‘g‘ri qiymatlarni avtomatik o‘chirib yuborishdan oldin ularning sonini o‘lchaymiz. Outlierlar alohida tekshiriladi; ular har doim ham xato degani emas.


In [ ]:
quality = {
    'rows_before': len(df),
    'duplicate_rows': int(df.duplicated().sum()),
    'missing_price': int(df['selling_price'].isna().sum()),
    'non_positive_price': int((df['selling_price'] <= 0).sum()),
    'invalid_year': int((~df['year'].between(1980, 2035)).sum()),
    'negative_mileage': int((df['km_driven'] < 0).sum()),
}
display(pd.Series(quality, name='count').to_frame())

df_clean = df.drop_duplicates().copy()
df_clean = df_clean.dropna(subset=['make', 'model', 'year', 'selling_price', 'km_driven'])
df_clean = df_clean[
    df_clean['year'].between(1980, 2035)
    & df_clean['selling_price'].gt(0)
    & df_clean['km_driven'].ge(0)
]
df_clean = df_clean.reset_index(drop=True)
print(f'Qatorlar: {len(df):,} -> {len(df_clean):,}; olib tashlangan: {len(df)-len(df_clean):,}')


## 5. Descriptive statistics — narx va probeg

O‘rtacha qiymatni median bilan birga ko‘ramiz. Narx taqsimoti odatda o‘ng tomonga cho‘zilgan bo‘lishi mumkin, shuning uchun median muhim.


In [ ]:
display(df_clean[['year', 'selling_price', 'km_driven']].describe().T)
print('Narx medianasi (INR):', f"{df_clean['selling_price'].median():,.0f}")
print('Narx o‘rtachasi (INR):', f"{df_clean['selling_price'].mean():,.0f}")


## 6. Grafiklar — toza va o‘qilishi oson vizualizatsiya

Har bir grafik `reports/figures/` papkasiga PNG formatida saqlanadi. Grafiklar ma’lumotni tushunishga xizmat qiladi; ular bezak uchun emas.


In [ ]:
def save_figure(filename):
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=220, bbox_inches='tight')
    plt.show()

# 6.1 Narx taqsimoti
plt.figure(figsize=(11, 5))
plt.hist(df_clean['selling_price'], bins=40, edgecolor='white')
plt.title('Avtomobil narxlari taqsimoti')
plt.xlabel('Narx (INR)')
plt.ylabel('Avtomobillar soni')
save_figure('01_price_distribution.png')

# 6.2 Ishlab chiqarilgan yil bo‘yicha narx
plt.figure(figsize=(11, 5))
plt.scatter(df_clean['year'], df_clean['selling_price'], alpha=0.45)
plt.title('Ishlab chiqarilgan yil va sotuv narxi')
plt.xlabel('Ishlab chiqarilgan yil')
plt.ylabel('Narx (INR)')
save_figure('02_year_vs_price.png')

# 6.3 Probeg va narx
plt.figure(figsize=(11, 5))
plt.scatter(df_clean['km_driven'], df_clean['selling_price'], alpha=0.4)
plt.title('Probeg va sotuv narxi')
plt.xlabel('Probeg (km)')
plt.ylabel('Narx (INR)')
save_figure('03_mileage_vs_price.png')


In [ ]:
# 6.4 3D interaktiv grafik: yil, probeg va narx
sample_3d = df_clean.sample(min(1500, len(df_clean)), random_state=42)
fig = px.scatter_3d(
    sample_3d,
    x='year', y='km_driven', z='selling_price',
    color='fuel' if 'fuel' in sample_3d.columns else None,
    hover_data=['make', 'model'],
    title='3D: avtomobil yili, probegi va narxi',
    labels={'year': 'Ishlab chiqarilgan yil', 'km_driven': 'Probeg (km)', 'selling_price': 'Narx (INR)'}
)
fig.update_layout(template='plotly_white', height=700, scene=dict(xaxis_title='Yil', yaxis_title='Probeg (km)', zaxis_title='Narx (INR)'))
fig.show()
fig.write_html(str(FIGURES_DIR / '04_vehicle_price_3d.html'))


In [ ]:
# 6.5 Marka bo‘yicha median narx — eng ko‘p uchraydigan 15 marka
brand_summary = (df_clean.groupby('make', as_index=False)
                 .agg(median_price=('selling_price', 'median'), listings=('selling_price', 'size'))
                 .query('listings >= 5')
                 .sort_values('median_price', ascending=False)
                 .head(15)
                 .sort_values('median_price'))
plt.figure(figsize=(11, 7))
plt.barh(brand_summary['make'], brand_summary['median_price'])
plt.title('Eng ko‘p kuzatilgan markalar: median narx')
plt.xlabel('Median narx (INR)')
plt.ylabel('Marka')
save_figure('05_brand_median_price.png')


## 7. Kategorik ustunlarni tekshirish

Kam uchraydigan kategoriyalar keyingi bosqichda modelning umumlashtirish qobiliyatiga ta’sir qilishi mumkin.


In [ ]:
for col in ['fuel', 'seller_type', 'transmission', 'owner']:
    if col in df_clean.columns:
        print(f'\n--- {col} ---')
        display(df_clean[col].value_counts(dropna=False).head(15).to_frame('count'))


## 8. Standartlashtirilgan faylni saqlash

Bu fayl INR valyutasida. Uni AutoPrice’ning UZS trening pipeline’iga yubormang. U alohida tajriba datasetidir.


In [ ]:
output_path = PROCESSED_DIR / 'cardekho_india_audit.csv'
df_clean.to_csv(output_path, index=False)
print('Saqlangan fayl:', output_path.resolve())
print('Yakuniy dataset hajmi:', df_clean.shape)
print('Valyuta:', df_clean['currency'].unique().tolist())


## 9. Birinchi bosqich yakuni — yozma xulosa

Quyidagi savollarga grafik va statistik natijalarga qarab javob yozing:

1. Datasetda nechta qator va ustun bor?
2. Qaysi ustunda missing value eng ko‘p?
3. Narx taqsimoti simmetrikmi yoki o‘ngga/chapga qiyshayganmi?
4. Probeg oshganda narx odatda qanday o‘zgaradi?
5. Qaysi marka median narxi yuqori va buning sababi nima bo‘lishi mumkin?
6. Qaysi ustunlar narxni bashorat qilishda foydali bo‘lishi mumkin?
7. Ushbu datasetni O‘zbekiston narxini prognozlashda nima sababdan bevosita ishlata olmaymiz?

**Keyingi notebook:** `02_eda_and_feature_study.ipynb` — chuqur EDA, outlier tahlili, narxga ta’sir qiluvchi omillar va gipotezalar.
